# ETL — VISp inhibitory Patch-seq: Cell Features

Writes 46 `CellFeatureDefinition` rows, one `CellFeatureSet` (`inh_visp_morph_features`), the wide-form morphology feature parquet, and one `CellFeatureMatrix` pointer. Also registers any cell ids present in the wide-form CSV but absent from the `DataItem` table (i.e., cells not in the original `_01` source CSV). Prerequisite: `etl_visp_inh_patchseq_01_dataset_dataitem.ipynb` (`project_id="visp_inh_patchseq"`, `dataset_id="visp_inh_patchseq"`).

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import pyarrow as pa
from deltalake import write_deltalake

from connects_common_connectivity.io.arrow_utils import (
    attach_linkml_metadata,
    build_arrow_schema,
    build_cell_feature_matrix_schema,
    models_to_table,
)
from connects_common_connectivity.models import (
    CellFeatureDefinition,
    CellFeatureMatrix,
    CellFeatureSet,
    DataItem,
    DataItemDataSetAssociation,
    Unit,
)
from connects_common_connectivity.config import output_root
from connects_common_connectivity.io import write_models


In [2]:
DEFS_CSV       = "/data/visp-features-and-mapping/inh_visp_patchseq_morph_feature_definitions.csv"
WIDE_CSV       = "/data/visp-features-and-mapping/inh_ivscc_features_wide_unnormalized.csv"
OUTPUT_ROOT    = output_root()
PROJECT_ID     = "visp_patchseq"
DATASET_ID     = "visp_inh_patchseq"
FEATURE_SET_ID = "inh_visp_morph_features"

print(f"OUTPUT_ROOT    : {OUTPUT_ROOT}")
print(f"PROJECT_ID     : {PROJECT_ID}")
print(f"DATASET_ID     : {DATASET_ID}")
print(f"FEATURE_SET_ID : {FEATURE_SET_ID}")

OUTPUT_ROOT    : ../scratch/em_patchseq_wnm_v2/
PROJECT_ID     : visp_patchseq
DATASET_ID     : visp_inh_patchseq
FEATURE_SET_ID : inh_visp_morph_features


## Prerequisite check

In [3]:
existing_dataitems = (
    pl.read_delta(OUTPUT_ROOT + "dataitem/")
    .filter(pl.col("project_id") == PROJECT_ID)
)
assert existing_dataitems.shape[0] > 0, (
    f"etl_visp_inh_patchseq_01 must be run first — no DataItem rows for project_id='{PROJECT_ID}'"
)
print(f"Prerequisite OK: {existing_dataitems.shape[0]} DataItem rows for project_id='{PROJECT_ID}'")

Prerequisite OK: 4287 DataItem rows for project_id='visp_patchseq'


## Register new cells from the wide CSV

Check which cell ids in the wide CSV are not yet in the `DataItem` table, register any new ones
via `append_new_dataitems`, and add `DataItemDataSetAssociation` rows for those new cells.

In [4]:
# Determine which wide CSV cells are not yet in the DataItem table.
existing_ids = set(existing_dataitems["id"].to_list())

wide_ids_df = pd.read_csv(WIDE_CSV, usecols=["specimen_id"])
all_wide_ids = [str(sid) for sid in wide_ids_df["specimen_id"]]
new_ids = [cid for cid in all_wide_ids if cid not in existing_ids]
print(f"Cells in wide CSV   : {len(all_wide_ids)}")
print(f"Already in DataItem : {len(all_wide_ids) - len(new_ids)}")
print(f"New to register     : {len(new_ids)}")

Cells in wide CSV   : 520
Already in DataItem : 400
New to register     : 120


In [5]:
if new_ids:
    n_di = write_models([DataItem(id=cid, name=cid, project_id=PROJECT_ID) for cid in new_ids], output_root=OUTPUT_ROOT).rows_written
    print(f"DataItems appended: {n_di}")

    schema_assoc = build_arrow_schema(DataItemDataSetAssociation)
    new_assoc_table = attach_linkml_metadata(
        models_to_table(
            [
                DataItemDataSetAssociation(dataitem_id=cid, dataset_id=DATASET_ID, project_id=PROJECT_ID)
                for cid in new_ids
            ],
            schema=schema_assoc,
        ),
        linkml_class="DataItemDataSetAssociation",
    )
    # mode="append" is safe here: new_ids only contains cells not yet in DataItem.
    # Re-runs skip this block (new_ids is empty), so no duplicate associations accumulate.
    write_deltalake(
        OUTPUT_ROOT + "dataitem_dataset_association/", new_assoc_table,
        mode="append", partition_by=["project_id"],
    )
    print(f"Associations appended: {len(new_ids)}")
else:
    print("No new cells to register — all already present.")

DataItems appended: 120
Associations appended: 120


In [6]:
# Verification
di_v = pl.read_delta(OUTPUT_ROOT + "dataitem/").filter(pl.col("project_id") == PROJECT_ID)
print(f"Total DataItems for project: {di_v.shape[0]}")
all_registered = set(di_v["id"].to_list())
assert all(cid in all_registered for cid in all_wide_ids), "Some wide CSV cells missing from DataItem"
assert di_v["id"].n_unique() == di_v.shape[0], "Duplicate ids in DataItem table"

assoc_v = (
    pl.read_delta(OUTPUT_ROOT + "dataitem_dataset_association/")
    .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("dataset_id") == DATASET_ID))
)
print(f"Associations for {DATASET_ID}: {assoc_v.shape[0]}")
assert all(cid in set(assoc_v["dataitem_id"].to_list()) for cid in all_wide_ids), \
    "Some wide CSV cells missing from DataItemDataSetAssociation"

Total DataItems for project: 4407


Associations for visp_inh_patchseq: 2879


## Load feature definitions

In [7]:
defs_df = pd.read_csv(DEFS_CSV)
print("Definitions shape:", defs_df.shape)
defs_df.head(3)

Definitions shape: (46, 6)


,id,description,unit,data_type,range_min,range_max
0,axon_bias_x,Difference in axon extent in the x-dimension (...,MICRONS_LENGTH,<f4,0.0,NaN
1,axon_bias_y,Difference in axon extent in the y-dimension (...,MICRONS_LENGTH,<f4,NaN,NaN
2,axon_depth_pc_0,First principal component of PCA performed on ...,NONE,<f4,NaN,NaN


## Write `CellFeatureDefinition` rows

In [8]:
feature_defs = []
for _, row in defs_df.iterrows():
    kwargs = dict(
        id=str(row["id"]),
        description=str(row["description"]),
        unit=str(row["unit"]),
        data_type=str(row["data_type"]),
        project_id=PROJECT_ID,
        feature_set_id=FEATURE_SET_ID,
    )
    if pd.notna(row["range_min"]):
        kwargs["range_min"] = float(row["range_min"])
    if pd.notna(row["range_max"]):
        kwargs["range_max"] = float(row["range_max"])
    feature_defs.append(CellFeatureDefinition(**kwargs))
result = write_models(feature_defs, output_root=OUTPUT_ROOT)
print(f"CellFeatureDefinition written: {result.rows_written} rows")

CellFeatureDefinition written: 46 rows


In [9]:
# Verification
cfd_v = (
    pl.read_delta(OUTPUT_ROOT + "cellfeaturedefinition/")
    .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("feature_set_id") == FEATURE_SET_ID))
)
print(cfd_v.shape); print(cfd_v.head(3))
assert cfd_v.shape[0] == len(feature_defs)
assert cfd_v["id"].n_unique() == len(feature_defs)

(46, 8)
shape: (3, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ id         ┆ descriptio ┆ unit       ┆ data_type ┆ range_min ┆ range_max ┆ project_i ┆ feature_s │
│ ---        ┆ n          ┆ ---        ┆ ---       ┆ ---       ┆ ---       ┆ d         ┆ et_id     │
│ str        ┆ ---        ┆ str        ┆ str       ┆ f64       ┆ f64       ┆ ---       ┆ ---       │
│            ┆ str        ┆            ┆           ┆           ┆           ┆ str       ┆ str       │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ axon_bias_ ┆ Difference ┆ MICRONS_LE ┆ <f4       ┆ 0.0       ┆ null      ┆ visp_patc ┆ inh_visp_ │
│ x          ┆ in axon    ┆ NGTH       ┆           ┆           ┆           ┆ hseq      ┆ morph_fea │
│            ┆ extent in  ┆            ┆           ┆           ┆           ┆           ┆ tures     │
│            ┆ t…         ┆            ┆           ┆           ┆     

## Write `CellFeatureSet`

In [10]:
feature_set = CellFeatureSet(
    id=FEATURE_SET_ID,
    description=(
        "Morphological features of inhibitory VISp Patch-seq neurons computed from "
        "reconstructed dendritic and axonal arbors. Used in MET-type analysis "
        "(multimodal electrophysiology, morphology, transcriptomics) to characterize "
        "inhibitory cell type diversity in mouse primary visual cortex."
    ),
    feature_definition_ids=[fd.id for fd in feature_defs],
    extraction_method="Computed via https://github.com/AllenInstitute/skeleton_keys.",
    project_id=PROJECT_ID,
)
result = write_models([feature_set], output_root=OUTPUT_ROOT)
print(f"CellFeatureSet written: {result.rows_written} rows")

CellFeatureSet written: 1 rows


In [11]:
# Verification
cfs_v = pl.read_delta(OUTPUT_ROOT + "cellfeatureset/").filter(pl.col("id") == FEATURE_SET_ID)
print(cfs_v.shape); print(cfs_v)
assert cfs_v.shape[0] == 1

(1, 5)
shape: (1, 5)
┌────────────────────┬────────────────────┬────────────────────┬───────────────────┬───────────────┐
│ id                 ┆ description        ┆ feature_definition ┆ extraction_method ┆ project_id    │
│ ---                ┆ ---                ┆ _ids               ┆ ---               ┆ ---           │
│ str                ┆ str                ┆ ---                ┆ str               ┆ str           │
│                    ┆                    ┆ list[str]          ┆                   ┆               │
╞════════════════════╪════════════════════╪════════════════════╪═══════════════════╪═══════════════╡
│ inh_visp_morph_fea ┆ Morphological      ┆ ["axon_bias_x",    ┆ Computed via http ┆ visp_patchseq │
│ tures              ┆ features of inhi…  ┆ "axon_bias_y",…    ┆ s://github.co…    ┆               │
└────────────────────┴────────────────────┴────────────────────┴───────────────────┴───────────────┘


## Load and write wide-form feature parquet

In [12]:
wide_df = pd.read_csv(WIDE_CSV)
print("Wide CSV shape:", wide_df.shape)

# Rename id column; convert int64 → str to match DataItem ids (values unchanged).
wide_df = wide_df.rename(columns={"specimen_id": "id"})
wide_df["id"] = wide_df["id"].astype(str)
wide_df["project_id"]     = PROJECT_ID
wide_df["feature_set_id"] = FEATURE_SET_ID

# Cast each feature column to its declared data_type.
for fd in feature_defs:
    if fd.id in wide_df.columns:
        wide_df[fd.id] = wide_df[fd.id].astype(np.dtype(fd.data_type))

wide_df.head(3)

Wide CSV shape: (520, 47)


,id,axon_bias_x,axon_bias_y,axon_depth_pc_0,axon_depth_pc_1,axon_depth_pc_2,axon_depth_pc_3,axon_depth_pc_4,axon_depth_pc_5,axon_emd_with_basal_dendrite,...,basal_dendrite_soma_percentile_x,basal_dendrite_soma_percentile_y,basal_dendrite_stem_exit_down,basal_dendrite_stem_exit_side,basal_dendrite_stem_exit_up,basal_dendrite_total_length,basal_dendrite_total_surface_area,soma_aligned_dist_from_pia,project_id,feature_set_id
0,601506507,180.833191,-249.830750,-255.225098,23.857132,-264.429962,-299.517761,-400.345398,28.557966,22.687988,...,0.137298,0.522512,0.000,0.500000,0.500000,2518.295654,7207.459961,357.159821,visp_patchseq,inh_visp_morph_features
1,601790961,25.481123,434.251068,-216.809891,-153.378464,-303.881836,-117.440689,241.134079,-7.542952,39.412388,...,0.480986,0.462676,0.000,0.666667,0.333333,4256.093750,11691.149414,663.103027,visp_patchseq,inh_visp_morph_features
2,601803754,42.650597,104.697845,1157.151978,3052.272949,-16.743603,996.874817,318.205719,-209.400940,16.735521,...,0.460057,0.333470,0.125,0.625000,0.250000,4108.235352,11384.542969,170.365067,visp_patchseq,inh_visp_morph_features


In [13]:
schema_wide = build_cell_feature_matrix_schema(feature_set, feature_defs, cell_index_column="id")
table_wide  = pa.Table.from_pandas(wide_df, schema=schema_wide, preserve_index=False)

write_deltalake(
    OUTPUT_ROOT + f"cellfeatures/{FEATURE_SET_ID}/", table_wide,
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}'",
    partition_by=["project_id", "feature_set_id"],
)
print("Wide-form parquet written:", table_wide.shape)

Wide-form parquet written: (520, 49)


In [14]:
# Verification
wide_v = (
    pl.read_delta(OUTPUT_ROOT + f"cellfeatures/{FEATURE_SET_ID}/")
    .filter(pl.col("project_id") == PROJECT_ID)
)
print(wide_v.shape); print(wide_v.head(3))
assert wide_v.shape[0] == len(wide_df)
assert wide_v["id"].n_unique() == len(wide_df), "Duplicate cell ids in wide table"

(520, 49)
shape: (3, 49)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ id        ┆ axon_bias ┆ axon_bias ┆ axon_dept ┆ … ┆ basal_den ┆ soma_alig ┆ project_i ┆ feature_ │
│ ---       ┆ _x        ┆ _y        ┆ h_pc_0    ┆   ┆ drite_tot ┆ ned_dist_ ┆ d         ┆ set_id   │
│ str       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ al_surfac ┆ from_pia  ┆ ---       ┆ ---      │
│           ┆ f32       ┆ f32       ┆ f32       ┆   ┆ e_a…      ┆ ---       ┆ str       ┆ str      │
│           ┆           ┆           ┆           ┆   ┆ ---       ┆ f32       ┆           ┆          │
│           ┆           ┆           ┆           ┆   ┆ f32       ┆           ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 601506507 ┆ 180.83319 ┆ -249.8307 ┆ -255.2250 ┆ … ┆ 7207.4599 ┆ 357.15982 ┆ visp_patc ┆ inh_visp │
│           ┆ 1         ┆ 5         ┆ 98        ┆   ┆ 61        ┆ 

## Write `CellFeatureMatrix` pointer

In [15]:
output_abs = Path(OUTPUT_ROOT).resolve()
cfm = CellFeatureMatrix(
    id=f"{PROJECT_ID}_{FEATURE_SET_ID}",
    feature_set_id=FEATURE_SET_ID,
    parquet_path=f"file://{output_abs}/cellfeatures/{FEATURE_SET_ID}/",
    cell_index_column="id",
    project_id=PROJECT_ID,
)
result = write_models([cfm], output_root=OUTPUT_ROOT)
print(f"CellFeatureMatrix written: {result.rows_written} rows")

CellFeatureMatrix written: 1 rows


In [16]:
# Verification
cfm_v = (
    pl.read_delta(OUTPUT_ROOT + "cellfeaturematrix/")
    .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("feature_set_id") == FEATURE_SET_ID))
)
print(cfm_v.shape); print(cfm_v)
assert cfm_v.shape[0] == 1

(1, 5)
shape: (1, 5)
┌────────────────────┬────────────────────┬────────────────────┬───────────────────┬───────────────┐
│ id                 ┆ feature_set_id     ┆ parquet_path       ┆ cell_index_column ┆ project_id    │
│ ---                ┆ ---                ┆ ---                ┆ ---               ┆ ---           │
│ str                ┆ str                ┆ str                ┆ str               ┆ str           │
╞════════════════════╪════════════════════╪════════════════════╪═══════════════════╪═══════════════╡
│ visp_patchseq_inh_ ┆ inh_visp_morph_fea ┆ file:///scratch/em ┆ id                ┆ visp_patchseq │
│ visp_morph_f…      ┆ tures              ┆ _patchseq_wn…      ┆                   ┆               │
└────────────────────┴────────────────────┴────────────────────┴───────────────────┴───────────────┘


## Summary

| Output path | Class | Rows |
|---|---|---|
| `dataitem/` | `DataItem` | +new cells from wide CSV (≤ 520 total, 120 new on first run) |
| `dataitem_dataset_association/` | `DataItemDataSetAssociation` | +new cells from wide CSV |
| `cellfeaturedefinition/` | `CellFeatureDefinition` | 46 |
| `cellfeatureset/` | `CellFeatureSet` | 1 (`inh_visp_morph_features`) |
| `cellfeatures/inh_visp_morph_features/` | wide parquet | 520 cells × 46 features |
| `cellfeaturematrix/` | `CellFeatureMatrix` | 1 |

`dataitem/` and `dataitem_dataset_association/` use `append_new_dataitems` / `mode="append"` scoped to new cells only — re-running is idempotent and never wipes rows from `etl_visp_inh_patchseq_01`. All other writes use `mode="overwrite"` with a scoped predicate.